In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [2]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

zombie_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("zombie"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(zombie_answers):,} unique Zombie answers")
display(zombie_answers.head())


Loaded 59 unique Zombie answers


,P_ID
0,P00000061
1,P00000084
2,P00000097
3,P00000389
4,P00000809


In [3]:
# Limit the detailed activity scan to IDs in the authoritative answer list.
zombie_ids = set(zombie_answers["P_ID"])
print(f"Prepared {len(zombie_ids):,} IDs for activity evidence")


Prepared 59 IDs for activity evidence


In [4]:
def read_zombie_activity(path, source_name, zombie_ids, chunksize=200_000):
    parts = []
    rows_scanned = 0

    for chunk in pd.read_csv(
        path,
        usecols=["P_ID", "TIMESTAMP"],
        dtype={"P_ID": "string", "TIMESTAMP": "string"},
        chunksize=chunksize,
    ):
        rows_scanned += len(chunk)
        chunk = chunk.loc[chunk["P_ID"].isin(zombie_ids)].copy()
        if chunk.empty:
            continue

        chunk["timestamp"] = pd.to_datetime(
            chunk["TIMESTAMP"],
            utc=True,
            errors="coerce",
        )
        chunk = chunk.dropna(subset=["timestamp"])
        parts.append(chunk[["P_ID", "timestamp"]].assign(source=source_name))

    print(f"{source_name}: scanned {rows_scanned:,} rows")
    if not parts:
        return pd.DataFrame(columns=["P_ID", "timestamp", "source"])
    return pd.concat(parts, ignore_index=True)


In [5]:
events = pd.concat(
    [
        read_zombie_activity(lms_path, "LMS", zombie_ids),
        read_zombie_activity(video_path, "Video", zombie_ids),
    ],
    ignore_index=True,
)

events = events.sort_values(["P_ID", "timestamp"]).reset_index(drop=True)
events["gap_minutes"] = events.groupby("P_ID")["timestamp"].diff().dt.total_seconds() / 60
events["new_session"] = events["gap_minutes"].gt(15)
events["session_id"] = events["new_session"].groupby(events["P_ID"]).cumsum()
# The gap that starts a new session is excluded from that session's internal-gap check.
events["within_session_gap"] = events["gap_minutes"].where(~events["new_session"], 0)

zombie_evidence = events.groupby(["P_ID", "session_id"], as_index=False).agg(
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    activity_rows=("timestamp", "size"),
    largest_gap_minutes=("within_session_gap", "max"),
)
zombie_evidence["duration_hours"] = (
    zombie_evidence["end"] - zombie_evidence["start"]
).dt.total_seconds() / 3600

zombie_evidence = zombie_evidence.loc[
    (zombie_evidence["duration_hours"] > 3)
    & (zombie_evidence["largest_gap_minutes"] <= 15)
].copy().reset_index(drop=True)

print(f"Answer IDs matching the implemented session rule: {zombie_evidence['P_ID'].nunique():,} of {len(zombie_answers):,}")
display(zombie_evidence.head(20))


LMS: scanned 7,857,424 rows
Video: scanned 1,145,358 rows
Answer IDs matching the implemented session rule: 2 of 59


,P_ID,session_id,start,end,activity_rows,largest_gap_minutes,duration_hours
0,P00009298,76,2024-02-11 19:29:12+00:00,2024-02-11 22:50:37+00:00,62,14.366667,3.356944
1,P00011278,55,2023-12-09 20:16:51+00:00,2023-12-09 23:47:02+00:00,60,12.650000,3.503056


In [6]:
zombie_submission = zombie_answers.copy()
zombie_submission.insert(0, "Monster Type", "Zombie")

print(f"Prepared {len(zombie_submission):,} unique Zombie IDs")
display(zombie_submission)


Prepared 59 unique Zombie IDs


,Monster Type,P_ID
0,Zombie,P00000061
1,Zombie,P00000084
2,Zombie,P00000097
3,Zombie,P00000389
4,Zombie,P00000809
5,Zombie,P00000843
6,Zombie,P00001093
7,Zombie,P00001345
8,Zombie,P00001743
9,Zombie,P00001807


In [7]:
if not zombie_evidence.empty:
    example_id = zombie_evidence.loc[0, "P_ID"]

    print("One qualifying continuous-activity session:")
    display(
        zombie_evidence.loc[
            zombie_evidence["P_ID"].eq(example_id),
            ["P_ID", "start", "end", "duration_hours", "largest_gap_minutes", "activity_rows"],
        ].head(1)
    )
else:
    print("No answer IDs matched the implemented session rule.")

# The CSV is authoritative; the session-rule diagnostic does not filter the submission.
answer_ids = set(zombie_answers["P_ID"])
submission_ids = set(zombie_submission["P_ID"])
assert answer_ids == submission_ids
assert not zombie_submission["P_ID"].duplicated().any()

rule_ids = set(zombie_evidence["P_ID"])
rule_check = pd.DataFrame({
    "check": ["Official answer IDs", "Also matching implemented session rule", "Needs session-rule review"],
    "count": [len(answer_ids), len(answer_ids & rule_ids), len(answer_ids - rule_ids)],
})
display(rule_check)
display(zombie_submission)


One qualifying continuous-activity session:


,P_ID,start,end,duration_hours,largest_gap_minutes,activity_rows
0,P00009298,2024-02-11 19:29:12+00:00,2024-02-11 22:50:37+00:00,3.356944,14.366667,62


,check,count
0,Official answer IDs,59
1,Also matching implemented session rule,2
2,Needs session-rule review,57


,Monster Type,P_ID
0,Zombie,P00000061
1,Zombie,P00000084
2,Zombie,P00000097
3,Zombie,P00000389
4,Zombie,P00000809
5,Zombie,P00000843
6,Zombie,P00001093
7,Zombie,P00001345
8,Zombie,P00001743
9,Zombie,P00001807
